In [ ]:
import sys
import numpy as np
import tensorflow as tf
import tensorflow.keras as K
from pathlib import Path
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

In [ ]:
ROOT_DIR=Path.cwd().parents[1]
sys.path.append(str(ROOT_DIR/"src"/"data_preprocessing"))
from normalize_fn import load

In [ ]:
hidden_layers=[1024,1024]
epochs=950
act_func=tf.nn.relu
input_dropout=0.2
hidden_dropout=0.5
learning_rate=0.0001
norm='norm'

In [ ]:
X_tr, X_val, _, _, y_tr, y_val, _, _ = load(norm=norm)

print("Training data shape:", X_tr.shape)
print("Validation data shape:", X_val.shape)
print("Training targets shape:", y_tr.shape)
print("Validation targets shape:", y_val.shape)

print("NaN in X_tr:", np.isnan(X_tr).any())
print("Inf in X_tr:", np.isinf(X_tr).any())
print("NaN in y_tr:", np.isnan(y_tr).any())
print("Inf in y_tr:", np.isinf(y_tr).any())

In [ ]:
model=Sequential()
for i,units in enumerate(hidden_layers):
    if i==0:
        model.add(Dense(
            units,
            input_shape=(X_tr.shape[1],),
            activation=act_func,
            kernel_initializer='he_normal'))
        if input_dropout>0:
            model.add(Dropout(float(input_dropout)))
    else:
        model.add(Dense(
            units, 
            activation=act_func, 
            kernel_initializer='he_normal'))
        if hidden_dropout>0:
            model.add(Dropout(float(hidden_dropout)))
model.add(Dense(
    1,activation='linear',
    kernel_initializer='he_normal'))


In [ ]:
optimizer=tf.keras.optimizers.SGD(learning_rate=learning_rate,momentum=0.5)
model.compile(loss='mean_squared_error',optimizer=optimizer)
model.summary()

In [ ]:
history = model.fit(
    X_tr,y_tr,
    validation_data=(X_val,y_val),
    epochs=epochs,
    batch_size=64,
    shuffle=True,
    verbose=1
)

In [ ]:
print("Final training loss:",history.history['loss'][-1])
print("Final validation loss:",history.history['val_loss'][-1])
model.save("DeepSynergy_final.h5")